<h1>Seeds</h1>

In [ ]:
#%pip install -q --upgrade pip

In [ ]:
!pip install unsloth "xformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 59.7 MB/s e

In [ ]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F

In [ ]:
import json
import re
import gc

In [ ]:
from huggingface_hub import login
login(token='##############')  # Replace with your actual token

In [ ]:
#from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "meta-llama/Meta-Llama-3-8B-instruct"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
def generate_seeds(num_seeds=20, seed=42):
    """Generates a list of random seeds.

    Args:
        num_seeds: The number of seeds to generate.
        seed: The initial seed for the random number generator (for reproducibility).

    Returns:
        A list of random integer seeds.
    """
    random.seed(seed)  # Set initial seed for reproducibility
    seeds = [random.randint(1, 100000) for _ in range(num_seeds)]
    return seeds

In [ ]:
def create_text_generation_pipeline(model, tokenizer, temperature=1.0):
    """
    Creates a text-generation pipeline with the given model and tokenizer.

    Args:
        model: The preloaded model for text generation.
        tokenizer: The corresponding tokenizer.
        temperature (float): Sampling temperature for generation (default: 1.0).
        max_new_tokens (int): Maximum number of tokens to generate (default: 1024).

    Returns:
        A transformers pipeline object for text generation.
    """
    return transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            trust_remote_code=True,
            pad_token_id=0,
            do_sample=True,
            temperature=1.0,
            max_new_tokens=1,
    )

# Example usage:
# pipe = create_text_generation_pipeline(model, tokenizer)


In [ ]:
seeds = generate_seeds(num_seeds=32)

In [ ]:
model,tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=8192,
        dtype=None,
        load_in_4bit=True,
    )
FastLanguageModel.for_inference(model)
pipe = create_text_generation_pipeline(model, tokenizer)

==((====))==  Unsloth 2025.6.8: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Device set to use cuda:0


#timeline

In [ ]:
timeline = pd.read_csv('/content/drive/MyDrive/centaur/horizon/horizon_data_test.csv')

In [ ]:
# Set up your tokenizer-to-token-ID mapping for decoding model output
letter_token_ids = {
    "I": tokenizer("I", add_special_tokens=False)['input_ids'][0],
    "H": tokenizer("H", add_special_tokens=False)['input_ids'][0],
}

In [ ]:
def extract_model_choice(raw_response: str) -> str:
    """
    Extracts choice ('I' or 'H') from model's raw response text.
    Handles JSON and loose formats robustly.
    """
    try:
        # First try direct JSON parsing
        response_data = json.loads(raw_response)
        choice = response_data.get("choice", "").strip().upper()
        if choice in {"I", "H"}:
            return choice

    except json.JSONDecodeError:
        # Fallback: Search for JSON pattern in text
        json_match = re.search(r'{\s*"choice"\s*:\s*"?(I|H)"?\s*}', raw_response, re.IGNORECASE)
        if json_match:
            response_data = json.loads(json_match.group().replace("'", '"'))  # normalize quotes
            choice = response_data.get("choice", "").strip().upper()
            if choice in {"I", "H"}:
                return choice

    # Final fallback: Find first standalone I or H
    char_match = re.search(r'\b[IiHh]\b', raw_response)
    if char_match:
        return char_match.group().upper()

    raise ValueError("No valid choice ('I' or 'H') found in model response")

In [ ]:
def format_forced_trials(forced_df) -> str:
    """Format forced (instructed) trials as readable prompt text."""
    lines = []
    for idx, (_, row) in enumerate(forced_df.iterrows(), start=1):
        lines.append(f"Trial {idx} (instructed): You were instructed to press {row['choice']} and received {row['reward']} points.")
    return "\n".join(lines)

def format_past_trials(past_df):
    """Format past free-choice trials for the prompt."""
    trials_text = []
    for _, row in past_df.iterrows():
        trials_text.append(f"Trial {row['trial']}(free): You chose <<{row['choice']}>> and get {row['reward']} points.")
    return trials_text

In [ ]:
def build_slot_prompt_llama(current_trial: int, past_trials: list, forced_choices: str, total_trials: int, game_number, total_games=320) -> str:
    """
    Builds a slot task prompt that:
    1. Maintains the meaning from build_game_intro
    2. Minimizes positional bias
    3. Encourages evidence-based exploration
    """
    # Randomize machine labels each game to avoid bias
    machines = ["H", "I"]
    random.shuffle(machines)
    label_a, label_b = machines

    # Show last 5 trials or placeholder
    recent_history_text = "\n".join(past_trials[-5:]) if past_trials else "No free choices yet."

    return f"""<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>

You are participating in multiple games ({total_games} total) with two slot machines (labeled {label_a} and {label_b}).
The two slot machines are different across different games.

**Game Structure:**
1. First 4 trials: Instructed choices (you're told which machine to pick)
2. Next 1 or 6 trials: Free choices (you decide)
3. After these free-choice trials, the game ends


**Key Facts:**
- Each choice earns points (pay attention to outcomes)
- Your goal: Maximize points across all games
- Each slot machine tends to pay out about the same amount of points on average

**Critical Instructions:**
1. NO positional bias: {label_a} ≠ left/default, {label_b} ≠ right/second
2. When evidence is weak, EXPLORE to gather information
3. Base decisions on observed outcomes only

<|eot_id|>

<|start_header_id|>user<|end_header_id|>
# Game {game_number} of {total_games} | Trial {current_trial} of {total_trials}

## Outcome History
**Instructed Trials:**
{forced_choices}

**Recent Free Choices:**
{recent_history_text}

## Your Decision
Which machine will you choose NEXT to maximize your points?
- Consider all observed outcomes
- Remember reward rates can change
- Avoid position-based choices
- Explore when uncertain

<|response_format|>
{{"choice": "{label_a}"}}  // OR {{"choice": "{label_b}"}}

<|critical_instructions|>
- Respond with VALID JSON ONLY
- Choose {label_a} or {label_b} - no other options
- Base choice on evidence, not habit

<|eot_id|>"""

In [ ]:
def fix_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    transformers.set_seed(seed)  # For Hugging Face models

In [ ]:
def generate(prompt, pipe, model, tokenizer, human_choice=None):
    """
    Generates model choice and computes log-likelihood of human choice.

    Args:
        prompt (str or list): The task prompt shown to the model.
        pipe (transformers.pipeline): Text generation pipeline.
        model (transformers.PreTrainedModel): The language model.
        tokenizer (transformers.PreTrainedTokenizer): The tokenizer for the model.
        human_choice (str): 'H' or 'I', the actual human response (optional).

    Returns:
        choice (str): The model's predicted choice (H or I).
        log_likelihood (float or None): Log-likelihood of the human choice.
    """
    import torch

    # Ensure prompt is a string
    prompt_str = "".join(prompt) if isinstance(prompt, list) else prompt

    # Generate model output
    outputs = pipe(prompt_str)
    full_text = outputs[0]['generated_text']

    # Extract model's choice (you should define this function to parse model's JSON)
    choice = extract_model_choice(full_text)  # e.g., returns "H" or "I"

    log_likelihood = None
    if human_choice and human_choice in ["H", "I"]:
        # Construct expected full completion
        human_response = f'{{"choice": "{human_choice}"}}'
        full_input = prompt_str + human_response

        # Tokenize and compute log-likelihood of full human response
        inputs = tokenizer(full_input, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            log_likelihood = outputs.loss.item()

    return choice, log_likelihood

In [ ]:
# Modified simulation function
def simulate_participant_by_block(timeline_df, pipe, participant_id, model, tokenizer):
    """Simulates a participant with log-likelihood tracking"""
    all_rows = []

    participant_df = timeline_df[timeline_df['participant_id'] == participant_id]

    for block_num in sorted(participant_df['block'].unique()):
        block_df = participant_df[participant_df['block'] == block_num]

        for game in sorted(block_df['game'].unique()):
            game_df = block_df[block_df['game'] == game].sort_values('trial')
            reward_means = {"H": game_df["m1"].iloc[0], "I": game_df["m2"].iloc[0]}
            horizon = game_df["horizon"].iloc[0]
            info_condition = game_df["uc"].iloc[0]
            forced_df = game_df[game_df["type"] == "forced"]
            free_df = game_df[game_df["type"] == "free"]
            cumulative_reward = forced_df["reward"].sum()

            # Process forced trials
            for _, row in forced_df.iterrows():
                trial = row["trial"]
                is_free = False
                all_rows.append({
                    "participant_id": participant_id,
                    "block": block_num,
                    "game": game,
                    "horizon": horizon,
                    "info_condition": info_condition,
                    "reward_mean_H": reward_means["H"],
                    "reward_mean_I": reward_means["I"],
                    "trial_num": row["trial"],
                    "choice": row["choice"],
                    "reward": row["reward"],
                    "cumulative_reward": cumulative_reward,
                    "is_free": False,
                    "log_likelihood": None  # Forced trials have no LL
                })

            # Process free-choice trials
            for _, row in free_df.iterrows():
                logits_list, probs_list, pred_token_id, log_likelihood = None, None, None, None
                trial = row["trial"]
                is_free = True
                current_trial = row["trial"]
                past_forced_df = game_df[game_df["type"] == "forced"]
                past_free_df = game_df[(game_df["type"] == "free") & (game_df["trial"] < current_trial)]

                forced_trials_text = format_forced_trials(past_forced_df)
                free_trials_text = format_past_trials(past_free_df)
                prompt = build_slot_prompt_llama(
                    current_trial=current_trial,
                    past_trials=free_trials_text,
                    forced_choices=forced_trials_text,
                    total_trials=len(game_df),
                    game_number=game
                )
                #print(f"Prompt for trial {current_trial}:")
                #print(prompt)
                # Get human choice from data
                human_choice = row['choice']

                # Generate model choice and log-likelihood
                model_choice, log_likelihood = generate(
                    prompt, pipe, model, tokenizer, human_choice
                )


                reward = row["reward"]
                cumulative_reward += reward
                all_rows.append({
                    "participant_id": participant_id,
                    "block": block_num,
                    "game": game,
                    "horizon": horizon,
                    "info_condition": info_condition,
                    "reward_mean_H": reward_means["H"],
                    "reward_mean_I": reward_means["I"],
                    "trial_num": trial,
                    "is_free": is_free,
                    "choice": human_choice,
                    "model_choice": model_choice,
                    "reward": reward,
                    "cumulative_reward": cumulative_reward,
                    #"logits": logits_list,
                    #"probs": probs_list,
                    #"pred_token_id": pred_token_id,
                    "log_likelihood_of_human_choice": log_likelihood
                })
                print(f"Trial {trial}: Human {human_choice}, Model {model_choice}, LL:{log_likelihood}")

    print(f"✅ Simulated participant {participant_id}.")
    return pd.DataFrame(all_rows)

<h3>testing(can be skipped)</h3>

In [ ]:
# Create a copy of the timeline with 'game' renamed to 'game_number' if needed
#timeline_test = timeline[:100].copy()
# Run the simulation for participant 1
#result_test_4 = simulate_participant_by_block(timeline_test, pipe, 1, seeds)

In [ ]:
#result_test_4
#model_choice=result_test_4[result_test_4['is_free']==True]
#model_choice['choice'].value_counts()


In [ ]:
# Check if the sentence "you press and you get" exists in any prompt in result_test_4
#contains_sentence = result_test_4['prompt'].dropna().str.contains("you press <<H>>", case=False).any()
#print("Sentence found:", contains_sentence)

In [ ]:
# Print the prompt used for model choice in the first free trial of result_test_4
#first_free_trial = result_test_4[result_test_4['is_free'] == True][:100]
#for _, row in first_free_trial.iterrows():
    #print(f"Trial {row['trial_num_block']}:")
    #print("Prompt:", row['prompt'])
    #print("Model choice:", row['choice'])
    #print("Reward:", row['reward'])
    #print("Cumulative reward:", row['cumulative_reward'])

<h3>Centaur run </h3>

In [ ]:
participant_ids = timeline['participant_id'].unique()

In [ ]:
fix_seed(seeds[0])

In [ ]:
all_results=[]
for participant_id in participant_ids:
    print(f"\n🧠 Simulating participant {participant_id}")

    # Run simulation with model and tokenizer passed
    participant_data = timeline[timeline['participant_id'] == participant_id]
    result = simulate_participant_by_block(
        participant_data, pipe, participant_id, model, tokenizer
    )
    all_results.append(result)



🧠 Simulating participant 1
Trial 5: Human I, Model I, LL:-2.8536934852600098
Trial 5: Human H, Model I, LL:-2.868035316467285
Trial 6: Human I, Model H, LL:-2.8943214416503906
Trial 7: Human I, Model I, LL:-2.787259340286255
Trial 8: Human I, Model I, LL:-2.7042348384857178
Trial 9: Human I, Model H, LL:-2.620884418487549
Trial 10: Human H, Model I, LL:-2.5407052040100098
Trial 5: Human H, Model H, LL:-2.839832067489624
Trial 6: Human H, Model I, LL:-2.9043424129486084


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Trial 7: Human H, Model I, LL:-2.787339210510254
Trial 8: Human H, Model H, LL:-2.687199592590332
Trial 9: Human H, Model I, LL:-2.6195554733276367
Trial 10: Human H, Model H, LL:-2.5188546180725098
Trial 5: Human H, Model I, LL:-2.8674540519714355
Trial 6: Human H, Model H, LL:-2.8695168495178223
Trial 7: Human H, Model I, LL:-2.782299280166626
Trial 8: Human H, Model I, LL:-2.696403980255127
Trial 9: Human I, Model H, LL:-2.586256980895996
Trial 10: Human H, Model H, LL:-2.49910831451416
Trial 5: Human I, Model H, LL:-2.8552205562591553
Trial 6: Human H, Model H, LL:-2.896096706390381
Trial 7: Human I, Model I, LL:-2.7964344024658203
Trial 8: Human I, Model H, LL:-2.694721221923828
Trial 9: Human I, Model H, LL:-2.617016315460205
Trial 10: Human H, Model I, LL:-2.5420002937316895
Trial 5: Human I, Model I, LL:-2.882051706314087
Trial 6: Human I, Model I, LL:-2.9046342372894287
Trial 7: Human H, Model I, LL:-2.807283639907837
Trial 8: Human I, Model I, LL:-2.703885555267334


KeyboardInterrupt: 

In [ ]:
all__results = pd.concat(all_results, ignore_index=True)

In [ ]:
all_results_df =pd.DataFrame(all__results)

In [ ]:
all_results_df

In [ ]:
# Sum of log likelihood for each participant
#sum_log_likelihood = all_results_df.groupby('participant_id')['log_likelihood_of_human_choice'].sum().reset_index()
#print("Sum of log likelihood for each participant:")
#sum_log_likelihood

In [ ]:
#negative_sum_log_likelihood = sum_log_likelihood.copy()
#negative_sum_log_likelihood['log_likelihood_of_human_choice'] = -negative_sum_log_likelihood['log_likelihood_of_human_choice']
#negative_sum_log_likelihood

In [ ]:
#import pandas as pd
# Number of free trials for each participant
#free_trial_counts = all_results_df[all_results_df['is_free'] == True].groupby('participant_id').size().reset_index(name='free_trial_count')
#print("\nNumber of free trials for each participant:")
#free_trial_counts

In [ ]:
# Merge sum of log likelihood and free trial counts
#merged_df = pd.merge(negative_sum_log_likelihood, free_trial_counts, on='participant_id')
# Calculate the average log likelihood per free trial
#merged_df['average_log_likelihood_per_free_trial'] = merged_df['log_likelihood_of_human_choice'] / merged_df['free_trial_count']
#print("\nAverage log likelihood per free trial for each participant:")
#merged_df[['participant_id', 'average_log_likelihood_per_free_trial']]